Here is the start of the project. First, I import the data 

In [2]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def fetch_titanic_data():
    tarball_path = Path('datasets/titanic.tgz')
    if not tarball_path.is_file():
        Path('datasets').mkdir(parents=True, exist_ok=True)
        url = 'https://homl.info/titanic.tgz'
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as titanic_tarball:
            titanic_tarball.extractall(path='datasets', filter='data')
    return (pd.read_csv(Path('datasets/titanic/test.csv')), pd.read_csv(Path('datasets/titanic/train.csv')))

test_data, train_data = fetch_titanic_data()

Let's have a look at what we are dealing with

In [ ]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


Cabin column has 77% of the data missing, it is worth getting rid of

In [11]:
train_data = train_data.drop(columns='Cabin')
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


Let's inspect the non-integer columns

In [16]:
print()
print(train_data['Name'].value_counts()) 
print()
print(train_data['Sex'].value_counts()) 
print()
print(train_data['Ticket'].value_counts()) 
print()
print(train_data['Embarked'].value_counts()) 



Name
Braund, Mr. Owen Harris                                1
Cumings, Mrs. John Bradley (Florence Briggs Thayer)    1
Heikkinen, Miss. Laina                                 1
Futrelle, Mrs. Jacques Heath (Lily May Peel)           1
Allen, Mr. William Henry                               1
                                                      ..
Montvila, Rev. Juozas                                  1
Graham, Miss. Margaret Edith                           1
Johnston, Miss. Catherine Helen 'Carrie'               1
Behr, Mr. Karl Howell                                  1
Dooley, Mr. Patrick                                    1
Name: count, Length: 891, dtype: int64

Sex
male      577
female    314
Name: count, dtype: int64

Ticket
347082             7
1601               7
CA. 2343           7
3101295            6
CA 2144            6
                  ..
SOTON/OQ 392076    1
211536             1
112053             1
111369             1
370376             1
Name: count, Length: 681, dtyp

- Names are unique strings, not sure how a name would be useful for predicting survivability, but will leave it for now
- Sex is a binary value, which would be useful for converting into 0 and 1
- Ticket value is also a (mostly) unique string, although some consist of integers. This column may be useful, but I don't know how to adapt it for the algorithm
- Embarked is a categorical value, that needs to be converted into 1, 2 and 3 or into a one hot